# 005 — SQL Runner with CodeMirror · 데모

Jupyter 노트북에서 동작하는 single-file SQL 편집기 + 실행 위젯.

| 단계 | 무엇을 보나 |
|---|---|
| 1️⃣ 환경 준비 | 임시 SQLite DB + 4개 테이블 |
| 2️⃣ `runner.show()` | CodeMirror 에디터 + 자동완성 + ▶ 실행 UI |
| 3️⃣ `runner.history()` | 날짜 sidebar + SQL/전체 [📋 복사] · 새 세션에서도 자동으로 이어짐 |
| 4️⃣ `on_execute` 콜백 직접 등록 | sqlite/list/dict/REST 등 임의 백엔드 |
| 5️⃣ 대량 스키마 stress | 사내 금융사급 113 테이블 / 1,800+ 컬럼 |


---

## 1️⃣ 환경 준비 — DB + runner

`with_sqlite` 헬퍼로 한 줄에 셋업. 각 컬럼에는 `doc` 으로 한국어 설명을 덧붙여 추천 popup / 트리 hover 에 노출되게 합니다.


In [1]:
import os, sys, sqlite3, tempfile
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from sql_codemirror import SQLRunnerCM

# ── 임시 DB + 데모 데이터 ──
db_path = os.path.join(tempfile.gettempdir(), 'sql_codemirror_demo.db')
if os.path.exists(db_path):
    os.remove(db_path)
with sqlite3.connect(db_path) as conn:
    conn.executescript("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY, name TEXT, email TEXT,
        region TEXT, plan_type TEXT, signup_at TIMESTAMP
    );
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY, user_id INTEGER REFERENCES users(id),
        amount REAL, status TEXT, ordered_at TIMESTAMP
    );
    CREATE TABLE products (
        id INTEGER PRIMARY KEY, sku TEXT UNIQUE,
        name TEXT, category TEXT, price REAL, stock INTEGER
    );
    CREATE TABLE events (
        id INTEGER PRIMARY KEY, user_id INTEGER,
        event_type TEXT, value REAL, occurred_at TIMESTAMP
    );
    INSERT INTO users VALUES
      (1,'김알리스','alice@ex.com','서울','pro','2026-01-10'),
      (2,'이밥','bob@ex.com','부산','free','2026-02-15'),
      (3,'박찰리','charlie@ex.com','서울','enterprise','2026-03-01'),
      (4,'최다나','dana@ex.com','대전','pro','2026-03-12'),
      (5,'정에반','evan@ex.com','서울','free','2026-04-02');
    INSERT INTO orders VALUES
      (1,1,15000,'paid','2026-04-01'),(2,1,8500,'paid','2026-04-05'),
      (3,1,93000,'paid','2026-04-12'),(4,2,29000,'paid','2026-04-08'),
      (5,3,15000,'cancelled','2026-04-10'),(6,4,117000,'paid','2026-04-15');
    INSERT INTO products VALUES
      (1,'SKU-001','노트북','electronics',1850000,12),
      (2,'SKU-002','마우스','electronics',45000,128);
    """)

runner = SQLRunnerCM.with_sqlite(db_path)
# 컬럼별 한국어 description (자동 추출된 type 위에 보강)
runner.add_table('users', [
    {'name':'id','type':'INTEGER','doc':'PK'},
    {'name':'name','type':'TEXT','doc':'표시 이름'},
    {'name':'email','type':'TEXT','doc':'로그인용 이메일'},
    {'name':'region','type':'TEXT','doc':'거주 지역'},
    {'name':'plan_type','type':'TEXT','doc':'free / pro / enterprise'},
    {'name':'signup_at','type':'TIMESTAMP','doc':'가입 시각'},
], description='사용자 마스터')
runner.add_table('orders', [
    {'name':'id','type':'INTEGER','doc':'PK'},
    {'name':'user_id','type':'INTEGER','doc':'FK → users.id'},
    {'name':'amount','type':'REAL','doc':'결제 금액 KRW'},
    {'name':'status','type':'TEXT','doc':'paid · cancelled · pending'},
    {'name':'ordered_at','type':'TIMESTAMP','doc':'주문 시각'},
], description='주문 내역')

print(f'✓ DB · runner 준비 완료 ({len(runner.tables)} 테이블)')


✓ DB · runner 준비 완료 (4 테이블)


---

## 2️⃣ `runner.show()` — 메인 UI

CodeMirror 에디터 + 좌측 entity 트리 + 컨텍스트 자동완성 + ▶ 실행 버튼이 한 셀에 통합됩니다.

**시도해보세요**:
- `Cmd/Ctrl + Enter` → ▶ 실행
- `SELECT * FROM ` 입력 → popup 자동완성에 4 테이블 후보
- `SELECT u.id, u.name FROM users u WHERE u.` → users 컬럼만 매치
- 좌측 entity 트리 → 테이블/컬럼 클릭 시 커서 위치에 인서트
- 잘못된 SQL (예: 괄호 미닫힘) → 빨간 ❌ 메시지 + ▶ 실행 버튼 disabled


In [2]:
runner.set_query(
    "-- ▶ 실행 (Cmd/Ctrl+Enter) · Ctrl+Space 자동완성\n"
    "SELECT u.name, u.region, COUNT(o.id) AS n_orders, SUM(o.amount) AS total\n"
    "FROM users u LEFT JOIN orders o ON o.user_id = u.id\n"
    "WHERE o.status = 'paid'\n"
    "GROUP BY u.id ORDER BY total DESC NULLS LAST;"
)
runner.show()


---

## 3️⃣ `runner.history()` — 실행 이력 + 날짜 sidebar

▶ 실행을 누를 때마다 자동 누적. `runner.history()` 호출 시:

- 📅 좌측 날짜 sidebar (클릭 → 해당 날짜 entry 만)
- 📋 항목별 / 전체 SQL 복사 버튼
- 🔁 새 노트북 세션에서도 자동으로 이어짐
- 🗑 [history 비우기] 버튼 (두 번 클릭 확인) — `runner.history()` 위에 표시


In [3]:
# 데모용 — 5번 실행해 history 채우기 (실제로는 ▶ 실행 클릭이 자동 누적)
import time
demo_queries = [
    "SELECT COUNT(*) AS total_users FROM users",
    "SELECT region, COUNT(*) AS n FROM users GROUP BY region",
    ("SELECT u.name, SUM(o.amount) AS total\n"
     "  FROM users u JOIN orders o ON o.user_id = u.id\n"
     "  WHERE o.status = 'paid'\n"
     "  GROUP BY u.id ORDER BY total DESC LIMIT 5"),
    "SELECT * FROM products WHERE price > 50000",
    "SELECT * FROM nonexistent_table",   # 의도적 에러
]
for q in demo_queries:
    try:
        runner.execute(q)
    except Exception:
        pass   # 에러도 history 에 자동 누적
    time.sleep(0.4)

print(f"누적: {len(runner.history)} 건")


누적: 24 건


In [5]:
runner.history()


---

## 4️⃣ `on_execute` 콜백 직접 등록 — 임의 백엔드

`SQLRunnerCM.with_sqlite()` 는 SQLite + pandas 한정 헬퍼. 사내 SQL 엔진 / mock dict / **pandas DataFrame** / REST API 등 **임의 함수** 를 `on_execute=fn` 으로 주입하면 같은 위젯이 그 백엔드 위에서 동작합니다. 시그니처: `(sql: str) -> Any`.

| 사례 | 백엔드 |
|---|---|
| A | sqlite raw cursor (pandas 없이 list[dict]) |
| B | 메모리 mock 엔진 (DB 없이 dict 위에서) |
| C | **pandas DataFrame** (이미 메모리에 있는 DF 들을 SQL 로 query) |
| D | REST / 사내 엔진 wrapper |


In [ ]:
# 사례 A — sqlite raw cursor (pandas 미의존, 결과는 list[dict])
def run_raw(sql: str):
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql).fetchall()]

# 사례 B — 메모리 mock 엔진 (DB 없이 dict 위에서)
MEMORY = {
    'fruit': [{'id': 1, 'name': '사과', 'price': 3000},
              {'id': 2, 'name': '바나나', 'price': 1500}],
}
def run_memory(sql: str):
    print(f'[MOCK] {sql.strip()[:60]}')
    return MEMORY['fruit']

# 사례 C — pandas DataFrame 위에서 SQL (DataFrame → 임시 in-memory sqlite → pd.read_sql)
df_users = pd.read_sql('SELECT * FROM users', sqlite3.connect(db_path))
df_orders = pd.read_sql('SELECT * FROM orders', sqlite3.connect(db_path))

def run_df(sql: str):
    """이미 메모리에 있는 DataFrame 들을 SQL 로 query — 사내 캐시·중간 산출물에 유용."""
    with sqlite3.connect(':memory:') as mem:
        df_users.to_sql('users', mem, index=False)
        df_orders.to_sql('orders', mem, index=False)
        return pd.read_sql(sql, mem)

# 사례 D — REST/사내 엔진 wrapper (실 코드는 requests / 사내 SDK 사용)
def run_via_api(sql: str):
    # response = requests.post('http://internal.api/sql', json={'sql': sql})
    # return response.json()['rows']
    return [{'engine': 'internal', 'sql_received': sql, 'rows_returned': 0}]

# DataFrame 백엔드로 띄우기 (다른 사례도 on_execute=run_raw / run_memory / run_via_api 로 교체 가능)
runner_df = SQLRunnerCM(on_execute=run_df, history_dir=None)
runner_df.from_dict({
    'users':  ['id', 'name', 'email', 'region', 'plan_type', 'signup_at'],
    'orders': ['id', 'user_id', 'amount', 'status', 'ordered_at'],
})
runner_df.set_query(
    "-- DataFrame 위에서 SQL 실행 — JOIN/GROUP BY 모두 정상 동작\n"
    "SELECT u.region, COUNT(o.id) AS n_orders, SUM(o.amount) AS total\n"
    "FROM users u LEFT JOIN orders o ON o.user_id = u.id\n"
    "GROUP BY u.region ORDER BY total DESC NULLS LAST;"
)ㅠ
runner_df.show()


---

## 5️⃣ 대량 스키마 stress — 113 테이블 (사내 금융사 시뮬)

좌측 entity 패널은 위젯 객체 대신 단일 HTML 한 덩어리로 렌더하므로 수천 개 entity 도 첫 렌더 지연이 거의 없습니다. **검색창** 에 입력하면 JS 자체 필터로 즉석 매치 (Python round-trip 0).

**검색 시도**: `customer` (9개 테이블) · `amount` (37개 컬럼) · `kyc` · `portfolio_`


In [ ]:
import random
random.seed(42)

def fake_engine(sql: str):
    return [{"row_id": i, "value": f"val_{i:03d}", "score": i * 1.5}
            for i in range(20)]

SCHEMA_GROUPS = {
    "고객": ["customers","customer_addresses","customer_contacts",
            "customer_documents","customer_consents","customer_segments",
            "customer_relationships","customer_notes","kyc_records",
            "beneficial_owners","customer_risk_profiles"],
    "계좌": ["accounts","account_holders","account_balances",
            "account_statements","account_transactions","account_holds",
            "account_overdrafts","account_fees","account_limits",
            "account_types","account_alerts","savings_accounts",
            "checking_accounts","credit_accounts","investment_accounts"],
    "거래": ["transactions","trade_executions","settlements","clearings",
            "wire_transfers","internal_transfers","fee_transactions",
            "currency_exchanges","dividend_payments","interest_payments",
            "transaction_journal","pending_transactions","reversals"],
    "주문": ["orders","order_executions","order_routing","order_book",
            "order_states","order_amendments","order_cancellations",
            "basket_orders","algorithmic_orders","dark_pool_orders",
            "smart_routing_logs"],
    "상품/증권": ["instruments","equity_securities","fixed_income_securities",
                "derivatives","options","futures","swaps","mutual_funds",
                "etfs","structured_products","instrument_master",
                "instrument_prices","instrument_dividends","corporate_actions"],
    "포트폴리오": ["portfolios","portfolio_holdings","portfolio_allocations",
                "portfolio_returns","portfolio_metrics","portfolio_constraints",
                "portfolio_rebalances","portfolio_risk","portfolio_benchmarks"],
    "컴플라이언스": ["aml_screenings","sanctions_lists","watchlists",
                "suspicious_activities","compliance_alerts",
                "regulatory_filings","audit_trails","ofac_screenings",
                "pep_records","compliance_cases"],
    "분석/이벤트": ["events","page_views","sessions","user_journeys",
                "funnel_steps","conversion_events","click_streams",
                "error_events","performance_metrics","ab_test_assignments"],
    "참조데이터": ["currencies","exchange_rates","countries","business_calendars",
                "holidays","market_data_sources","reference_indices",
                "sector_classifications","gics_codes","isin_database"],
    "시스템": ["system_users","system_roles","system_permissions",
            "system_sessions","api_keys","audit_logs","system_logs",
            "error_logs","access_logs","system_configuration"],
}

NUMERIC = ["amount","balance","fee","rate","score","quantity","volume",
           "weight","factor","threshold","premium","discount","tax",
           "interest_amount","principal","yield_rate","spread","ratio",
           "percentage","market_value","book_value","nav","price","unit_cost"]
TEXT = ["name","code","description","category","type","status","label",
        "tag","notes","comment","reason","source","channel","method",
        "currency_code","country_code","language_code","external_id",
        "reference_no","approval_code"]
DATE = ["created_at","updated_at","deleted_at","effective_date",
        "expiration_date","settled_at","executed_at","approved_at",
        "submitted_at","processed_at","cancelled_at","valued_at"]
INT = ["version","sequence_no","priority","max_retries","attempt_count",
       "row_count","line_no"]
FK = ["customer","account","order","transaction","instrument","portfolio",
      "product","user"]

def gen_cols(tname, count):
    cols = [{"name":"id","type":"INTEGER","doc":"PK"}]
    used = {"id"}
    for prefix in FK:
        if prefix in tname.split("_") and not tname.startswith(prefix):
            fk = f"{prefix}_id"
            if fk not in used:
                cols.append({"name":fk,"type":"INTEGER","doc":f"FK → {prefix}s.id"})
                used.add(fk)
    pools = [(NUMERIC,"REAL"),(TEXT,"TEXT"),(DATE,"TIMESTAMP"),(INT,"INTEGER")]
    while len(cols) < count:
        pool, ctype = random.choice(pools)
        cand = random.choice(pool)
        if cand not in used:
            used.add(cand)
            cols.append({"name":cand,"type":ctype,"doc":f"{cand} ({ctype.lower()})"})
    return cols

big = SQLRunnerCM(on_execute=fake_engine, history_dir=None)
total_t = total_c = 0
for group, tables in SCHEMA_GROUPS.items():
    for t in tables:
        n = random.randint(8, 25)
        big.add_table(t, gen_cols(t, n), description=f"[{group}] {t.replace('_',' ')}")
        total_t += 1; total_c += n

print(f"✓ {len(SCHEMA_GROUPS)} 도메인 · {total_t} 테이블 · {total_c} 컬럼 = {total_t+total_c} entities")
big.set_query("-- 검색창에 customer / amount / kyc / portfolio_ 입력해보세요\n"
              "SELECT * FROM customers LIMIT 20;")
big.show()


---

## 6️⃣ 다중 schema — 사이드바 그룹 + schema-first 자동완성 + 실제 ATTACH 실행

같은 이름의 테이블이 여러 환경 (`main` / `staging` / `analytics`) 에 공존하는 상황을 SQLite **`ATTACH DATABASE`** 로 진짜 실행합니다 — 각 환경을 별도 SQLite 파일로 만들어 두고 `on_execute` 콜백 안에서 ATTACH 하면 `SELECT * FROM staging.users` 같은 schema-qualified 쿼리가 실제로 결과를 돌려줍니다.

**한 셀에서 확인할 수 있는 것**:

| 영역 | 무엇을 보나 |
|---|---|
| 좌측 사이드바 | `📁 main` / `📁 staging` / `📁 analytics` 3개 그룹. 헤더 클릭으로 접기/펼치기. 검색창에 `staging` → staging 그룹만 / `users` → 동명 테이블이 main·staging 양쪽에 매치 |
| 자동완성 popup (FROM 뒤) | 첫 줄에 schema 후보 (📁 main · 📁 staging · 📁 analytics) → 선택 시 `staging.` 자동 인서트 + popup 자동 재호출 → 그 schema 의 테이블만 |
| 동명 테이블 disambiguation | `users` 가 main·staging 양쪽에 있어 사이드바 클릭이 자동으로 `main.users` / `staging.users` 형태로 인서트 |
| 3-segment qualifier | `staging.users.` 입력 시 staging.users 의 컬럼 후보 |
| ▶ 실행 | 미리 작성된 multi-schema JOIN 쿼리 — main 사용자와 staging 테스트 사용자를 매칭한 결과 표가 출력됨 |

**시도해볼 쿼리** (에디터에 직접 입력해서 popup 흐름 체험):

```sql
SELECT u.name, COUNT(e.id) AS n_events
FROM main.users u
LEFT JOIN analytics.events e ON e.user_id = u.id
GROUP BY u.id;
```


In [ ]:
# 세 개 schema 를 별도 SQLite 파일로 만들고 on_execute 안에서 ATTACH
import os, sqlite3, tempfile

_tmp = tempfile.gettempdir()
main_db_path      = os.path.join(_tmp, 'sql_cm_demo_main.db')
staging_db_path   = os.path.join(_tmp, 'sql_cm_demo_staging.db')
analytics_db_path = os.path.join(_tmp, 'sql_cm_demo_analytics.db')
for _p in (main_db_path, staging_db_path, analytics_db_path):
    if os.path.exists(_p):
        os.remove(_p)

# ── main schema (운영 DB) ──
with sqlite3.connect(main_db_path) as _c:
    _c.executescript("""
    CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT, email TEXT, plan TEXT);
    INSERT INTO users VALUES
        (1,'김알리스','alice@ex.com','pro'),
        (2,'이밥','bob@ex.com','free'),
        (3,'박찰리','charlie@ex.com','enterprise');
    CREATE TABLE orders(id INTEGER PRIMARY KEY, user_id INTEGER, amount REAL, status TEXT);
    INSERT INTO orders VALUES
        (1,1,15000,'paid'),(2,1,8500,'paid'),
        (3,2,29000,'paid'),(4,3,117000,'paid'),(5,3,15000,'cancelled');
    """)

# ── staging schema (테스트 환경 DB) — 동명 users 테이블이지만 컬럼이 다름 ──
with sqlite3.connect(staging_db_path) as _c:
    _c.executescript("""
    CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT, tier TEXT, env TEXT);
    INSERT INTO users VALUES
        (1,'테스트유저A','beta','sandbox'),
        (2,'테스트유저B','alpha','canary');
    CREATE TABLE feature_flags(id INTEGER PRIMARY KEY, key TEXT, enabled INTEGER);
    INSERT INTO feature_flags VALUES
        (1,'new_dashboard',1),(2,'dark_mode',1),(3,'beta_search',0);
    """)

# ── analytics schema (이벤트/로그 DB) ──
with sqlite3.connect(analytics_db_path) as _c:
    _c.executescript("""
    CREATE TABLE page_views(id INTEGER PRIMARY KEY, user_id INTEGER, url TEXT, viewed_at TIMESTAMP);
    INSERT INTO page_views VALUES
        (1,1,'/home','2026-04-25 10:00'),(2,1,'/pricing','2026-04-25 10:02'),
        (3,2,'/home','2026-04-25 11:30'),(4,3,'/dashboard','2026-04-25 12:00');
    CREATE TABLE events(id INTEGER PRIMARY KEY, user_id INTEGER, event_type TEXT, value REAL);
    INSERT INTO events VALUES
        (1,1,'click',1),(2,1,'scroll',1),(3,1,'purchase',15000),
        (4,2,'click',1),(5,3,'click',1);
    """)

# on_execute — main DB 를 열고 staging / analytics 를 ATTACH (매 호출마다 새 connection — thread 안전)
def run_multi_schema(sql):
    with sqlite3.connect(main_db_path) as conn:
        conn.execute(f"ATTACH DATABASE '{staging_db_path}' AS staging")
        conn.execute(f"ATTACH DATABASE '{analytics_db_path}' AS analytics")
        try:
            import pandas as pd
            return pd.read_sql(sql, conn)
        except ImportError:
            conn.row_factory = sqlite3.Row
            return [dict(r) for r in conn.execute(sql).fetchall()]

# Runner — 사이드바 등록도 schema 별로 (from_sqlite 가 schema= 인자 수용)
multi = SQLRunnerCM(on_execute=run_multi_schema, history_dir=None)
multi.from_sqlite(main_db_path)                          # → schema='main' (기본)
multi.from_sqlite(staging_db_path,   schema='staging')   # → schema='staging'
multi.from_sqlite(analytics_db_path, schema='analytics') # → schema='analytics'

print('✓ 등록된 schemas:', sorted(multi.tables.keys()))
for sch in sorted(multi.tables):
    print(f'  📁 {sch}:', sorted(multi.tables[sch].keys()))

multi.set_query(
    "-- ▶ 실행 → 운영(main)의 사용자와 테스트(staging)의 동명 사용자를 id 로 매칭\n"
    "-- 좌측 사이드바: 📁 main / 📁 staging / 📁 analytics 3그룹 (헤더 클릭으로 접기)\n"
    "-- 새 줄에 'SELECT * FROM ' 까지 친 뒤 popup → schema 후보 먼저, 선택 시 자동 재호출\n"
    "SELECT m.id,\n"
    "       m.name AS prod_name,\n"
    "       m.plan,\n"
    "       s.name AS staging_name,\n"
    "       s.tier,\n"
    "       s.env\n"
    "FROM main.users m\n"
    "LEFT JOIN staging.users s ON s.id = m.id;"
)
multi.show()


---

## 정리

| 기능 | 사용법 |
|---|---|
| ▶ 실행 + 결과 표 렌더 | `runner.show()` → Cmd/Ctrl+Enter |
| 다음 셀 분석 | `runner.last_result` · `runner.last_query` · `runner.history[-1]` |
| 이력 시각화 + 복사 | `runner.history()` · `runner.history(n=10)` · `runner.history.to_markdown()` |
| 외부 호출 | `runner.execute(sql)` (내부에서 history 누적까지 자동) |
| 임의 백엔드 | `SQLRunnerCM(on_execute=fn)` — sqlite/list/dict/REST 모두 가능 |
| 사내 정책 검증 | `SQLRunnerCM(on_validate=fn)` — 잘못된 SQL 시 ▶ 실행 disabled |
| history 비우기 | `runner.history()` 위 [🗑 history 비우기] 버튼 (두 번 클릭 확인) · 또는 `runner.clear_history()` |

## 폐쇄망 친화 체크

| 항목 | 상태 |
|---|---|
| 외부 네트워크 / CDN | ❌ 없음 (모든 CSS/JS 인라인) |
| `<link href>` / `<script src>` | ❌ 없음 |
| 새 서버 / 포트 | ❌ 없음 |
| 바이너리 영속화 | ❌ 없음 (history 는 텍스트 JSONL) |
| 단일 반입 단위 | `sql_codemirror.py` 한 파일 |
